
*   Gold Standards
*   Use Chain-of-Thought to force the Judge to explain its reasoning.
* Load the Interview Transcript, JD, and CV, as a JSON file.
* Output a structured JSON report.




In [ ]:
import json
import openai # or your preferred LLM client

def evaluate_interview(json_file_path):
    # 1. Load the data
    with open(json_file_path, 'r') as f:
        data = json.load(f)

    transcript = data.get('transcript')
    job_description = data.get('job_description')
    candidate_cv = data.get('candidate_cv')

    # 2. Define the "Gold Standard" Rubric with CoT
    system_prompt = """
    You are a Senior HR Auditor specializing in Structured Psychological Interviews.
    Your goal is to evaluate if an AI Interview Agent conducted a high-quality, fair interview.

    You will be provided with the Job Description, the Candidate's CV, and the Transcript.

    EVALUATION CRITERIA:
    1. RELEVANCE: Did the agent ask questions aligned with the JD and CV?
    2. DEPTH: Did the agent ask follow-up questions based on the candidate's specific answers?
    3. BIAS MITIGATION: Did the agent avoid personal/protected topics and stick to job-related skills?

    REQUIRED OUTPUT FORMAT:
    You must think step-by-step. First, analyze the alignment, then provide a score (1-10) for each category.
    Return your final response in valid JSON format.
    """

    user_prompt = f"""
    ### JOB DESCRIPTION:
    {job_description}

    ### CANDIDATE CV:
    {candidate_cv}

    ### INTERVIEW TRANSCRIPT:
    {transcript}

    ### EVALUATION TASK:
    1. Analyze the transcript against the JD and CV.
    2. Provide reasoning for each score.
    3. Output JSON with keys: "reasoning", "scores": {{"relevance": 0, "depth": 0, "fairness": 0}}, "final_verdict": ""
    """

    # 3. Call the "Judge" LLM
    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_format={ "type": "json_object" }
    )

    return response.choices[0].message.content

# Example usage
# result = evaluate_interview('interview_data.json')
# print(result)

### JSON file fromat for the input:

In [ ]:
{
  "job_description": "We are looking for an NLP Engineer...",
  "candidate_cv": "Experience with BERT, PyTorch...",
  "transcript": [
    {"role": "agent", "content": "Tell me about your NLP experience."},
    {"role": "candidate", "content": "I built a RAG system..."},
    {"role": "agent", "content": "Interesting, how did you handle retrieval latency?"}
  ]
}